In [1]:
import numpy as np
import h5py
import pickle
from colossus.lss import peaks
from colossus.cosmology import cosmology

Illustris API request function


In [2]:
import requests
import time

baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":"c489f7f5a9e0e61f5060fd16e9fb975e"}

data_dir = './illustris_data/'

def get(path, params=None):
    max_retry = 10

    # make HTTP GET request to path
    for n in range(max_retry):
        try:
            r = requests.get(path, params=params, headers=headers)
            # raise exception if response code is not HTTP SUCCESS (200)
            r.raise_for_status()
            break
        except requests.HTTPError:
            if n < max_retry-1:
                time.sleep(min(2**n, 30))
                continue
            else:
                raise TimeoutError()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = data_dir + r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        f.close()
        return filename # return the filename string

    return r

In [ ]:
cosmo = cosmology.setCosmology('illustris')
h=cosmo.h

# Constants
k_B   = 1.381e-16    # erg/K
c_cgs = 2.998e10     # cm/s
eV    = 1.602e-12    # erg
keV   = 1.0e3 * eV   # erg
T_CMB = 2.725        # K
Mpc_cm = 3.086e24    # cm per Mpc

m_wdm = 1.0 * keV    # rest-mass energy of the WDM particle [erg]
z_nr = m_wdm/(k_B * T_CMB) - 1
t_nr = 1.3e6  # seconds (age of the Universe at z_nr)

lambda_fs_cm = c_cgs * t_nr * (1 + z_nr)
lambda_fs_Mpc = lambda_fs_cm/Mpc_cm
#lambda_fs = 2 # Mpc/h; characteristic free-streaming length for m_wdm=1keV, per [Andrew J. Long and Moira Venegas JCAP06(2025)043] 
M_suppressed = peaks.lagrangianM(lambda_fs_Mpc/h) # In units of M_sun/h
print(f"{M_suppressed:.3e}")

1.411e+08


In [11]:
def fs_suppress(primary_halo):
    """Remove subhalos from primary_halo that are below the WDM free-streaming suppression mass cutoff."""
    subhalos = get(primary_halo['meta']['url'],{'limit':primary_halo['child_subhalos']['count']})
    subhalo_list = subhalos['child_subhalos']['results']

    print(f"Group has {len(subhalo_list)} subhalos")
    n_suppressed = 0

    for i in reversed(range(len(subhalo_list))):
        subhalo = get(subhalo_list[i]['url'])
        subhalo_mass = subhalo['mass'] * 10e10 # In units of M_sun/h
        
        if subhalo_mass < M_suppressed:
            del subhalo_list[i]
            n_suppressed+=1

    print(f"{n_suppressed} subhalos suppressed by WDM free streaming")

    return subhalo_list
    

In [12]:
sim = get(baseUrl+'TNG50-1/')
snaps = get(sim['snapshots'])
primary_halo = get(snaps[-1]['url']+f'halos/59/')

In [ ]:
for halo_index in range(71,100):
    primary_halo = get(snaps[-1]['url']+f'halos/{halo_index}/')
    wdm_subhalos = fs_suppress(primary_halo)
    with open(f'./illustris_data/suppression/halo{halo_index}_wdm_fs_sup.pkl','wb') as file:
        pickle.dump(wdm_subhalos, file)

Group has 1067 subhalos
426 subhalos suppressed by WDM free streaming
Group has 1396 subhalos
615 subhalos suppressed by WDM free streaming
Group has 1180 subhalos
